<a href="https://colab.research.google.com/github/joryhh/Capstone-Project--Building-Agentic-AI-Systems/blob/main/Capstone_Main_Workflow.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Requirements Engineering Copilot

A multi-agent system that reviews a software requirements document and proposes
improvements, with a Product Owner approving every change before it is applied.

A **Supervisor** decides which specialist reviewers an input needs, the selected reviewers
run as workers, the Supervisor merges their findings, the Product Owner approves or rejects
each one, and an **Editor** applies only what was approved.

| Agent | Responsibility |
|---|---|
| Requirements Supervisor | Routes to reviewers, then merges and orders their findings |
| Completeness Reviewer | Capabilities in the project description with no requirement |
| Ambiguity Reviewer | Vague or subjective wording |
| Conflict Reviewer | Contradictions between requirements (RAG-grounded) |
| Testability & Standards Reviewer | No objective pass/fail criterion (RAG-grounded) |
| Requirements Editor | Applies approved changes only; decides nothing |

**Stack** — LangGraph Functional API (`@task` / `@entrypoint`), Groq `llama-3.3-70b-versatile`,
HuggingFace MiniLM embeddings, in-process vector store, LangSmith tracing.

### How to run

1. Add `GROQ_API_KEY` and `LANGCHAIN_API_KEY` to Colab Secrets (key icon, left sidebar) and
   enable **Notebook access** for both.
2. **Runtime → Run all.**

*Completed under the SDAIA Academy "Building Agentic AI Systems" program.*

## 1. Setup

In [1]:
# Chroma is unused (an in-process InMemoryVectorStore replaces it); removing it and its
# OpenTelemetry deps avoids a startup conflict. numpy is intentionally left at Colab's 2.x —
# downgrading it breaks every preinstalled package compiled against it.
!pip uninstall -y chromadb opentelemetry-api opentelemetry-sdk \
    opentelemetry-exporter-otlp opentelemetry-exporter-otlp-proto-grpc -q

!pip install -qU langchain langchain-groq langgraph langgraph-supervisor pydantic python-docx \
    langchain-huggingface sentence-transformers langchain-text-splitters

import os
os.environ["CHROMA_SERVER_NO_TELEMETRY"] = "1"
print("Dependencies installed.")

Dependencies installed.


In [2]:
# API keys, read from Colab Secrets — never hardcoded.
from google.colab import userdata

os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")

# The variable is LANGCHAIN_TRACING_V2. LANGSMITH_TRACING_V2 does not exist and fails
# silently: no trace, no error.
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_API_KEY"] = userdata.get("LANGCHAIN_API_KEY")
os.environ["LANGCHAIN_PROJECT"] = "capstone-requirements-reviewer"

print("Keys loaded. LangSmith project:", os.environ["LANGCHAIN_PROJECT"])

Keys loaded. LangSmith project: capstone-requirements-reviewer


In [3]:
# Console formatting, so every demo prints in the same shape.
import textwrap

_W = 78

def banner(title):
    print("\n" + "=" * _W); print(title.upper()); print("=" * _W)

def step(title):
    print(f"\n--- {title} " + "-" * max(0, _W - len(title) - 5))

def ok(msg):
    print(f"[PASS] {msg}")

def kv(label, value, width=24):
    print(f"  {label:<{width}}{value}")

def wrap(text, indent=9):
    return textwrap.fill(str(text), width=_W, subsequent_indent=" " * indent)

print("Display helpers ready.")

Display helpers ready.


## 2. Shared schemas

Every agent speaks in these three types. `Requirement` and `ReviewFinding` are the contract
between the reviewers and the Supervisor; `ApprovedChange` is the contract between the
human-in-the-loop step and the Editor.

In [4]:
from __future__ import annotations
from typing import Literal, Optional
from pydantic import BaseModel, Field


class Requirement(BaseModel):
    """A single parsed requirement."""
    id: str = Field(description="Stable identifier, e.g. 'REQ-001'")
    text: str = Field(description="The requirement statement")
    category: Optional[str] = Field(default=None, description="Optional grouping")


class ReviewFinding(BaseModel):
    """One issue raised by any reviewer, against one requirement."""
    requirement_id: str = Field(description="ID of the requirement this refers to")
    issue_type: Literal["missing", "ambiguous", "conflicting", "untestable"] = Field(
        description="missing: should exist but doesn't; ambiguous: vague wording; "
                    "conflicting: contradicts another; untestable: no pass/fail criterion"
    )
    severity: Literal["low", "medium", "high", "critical"] = Field(
        description="How much this would hurt the project if left unresolved"
    )
    reason: str = Field(description="One or two sentences explaining the issue")
    suggested_change: str = Field(description="A concrete, measurable rewrite or addition")


class ApprovedChange(BaseModel):
    """A Product Owner decision on one finding, consumed by the Editor."""
    requirement_id: str
    action: Literal["replace", "add", "apply_po_edit", "leave_unchanged"] = Field(
        description="replace: use suggested_change; add: insert a new requirement; "
                    "apply_po_edit: use the PO's own wording; leave_unchanged: rejected"
    )
    edited_text: Optional[str] = Field(default=None, description="Set for every action except leave_unchanged")
    notes: Optional[str] = Field(default=None, description="PO rationale, optional")


print("Schemas frozen:", Requirement.__name__, ReviewFinding.__name__, ApprovedChange.__name__)

Schemas frozen: Requirement ReviewFinding ApprovedChange


## 3. Reliability policy

Defined before any agent, because `@task` binds its retry policy at decoration time — a
policy declared later in the notebook would never attach to anything.

Two error-handling strategies:

1. **Throttling.** Groq's free tier caps tokens per minute. Fanning out four reviewers at
   once produced a real `429 RateLimitError`, so one shared limiter spaces every call out.
2. **Retry with exponential backoff**, on transient errors only.

A third guard, `keep_known_requirement_ids`, filters reviewer output — see the comment for
why it is necessary.

In [5]:
from langchain_core.rate_limiters import InMemoryRateLimiter
from langgraph.types import RetryPolicy

# Strategy 1 — throttle: ~1 request every 5s, with a small burst allowance.
groq_rate_limiter = InMemoryRateLimiter(
    requests_per_second=0.2,
    check_every_n_seconds=0.1,
    max_bucket_size=2,
)


# Strategy 2 — retry transient failures only. A bad key or malformed request fails
# identically every time, so retrying it only delays the real error.
def _is_transient_error(exc: BaseException) -> bool:
    if type(exc).__name__ in {
        "RateLimitError", "APITimeoutError", "APIConnectionError",
        "InternalServerError", "ServiceUnavailableError",
        "TimeoutError", "ConnectionError",
    }:
        return True
    return getattr(exc, "status_code", None) in (429, 500, 502, 503, 504)


REVIEWER_RETRY = RetryPolicy(
    max_attempts=5,
    initial_interval=3.0,
    backoff_factor=2.0,
    max_interval=45.0,   # 3s, 6s, 12s, 24s — enough to clear a per-minute window
    jitter=True,
    retry_on=_is_transient_error,
)


def keep_known_requirement_ids(findings, requirements, label):
    """Drop findings citing a requirement id that is not under review.

    The knowledge base contains a sample SRS numbered REQ-101..REQ-110. Those ids reach
    the reviewer through the retrieved context, so a reviewer can return a well-formed
    finding about a requirement that exists only in the reference material. The Editor
    would then be asked to rewrite a requirement that is not in the document.
    'missing' findings are exempt: they carry GAP-nnn ids by design.
    """
    valid = {r.id for r in requirements}
    kept, dropped = [], []
    for f in findings:
        (kept if (f.issue_type == "missing" or f.requirement_id in valid) else dropped).append(f)
    if dropped:
        print(f"  [{label}] dropped {len(dropped)} finding(s) citing ids not under review: "
              f"{sorted({f.requirement_id for f in dropped})}")
    return kept


print(f"Throttle: {groq_rate_limiter.requests_per_second} req/s | "
      f"Retry: up to {REVIEWER_RETRY.max_attempts} attempts on transient errors")

Throttle: 0.2 req/s | Retry: up to 5 attempts on transient errors


## 4. Requirements Parser tool

A real tool: it reads `document_text` and extracts from it. Returns structured
`Requirement` records with stable IDs, ready for the reviewers.

In [6]:
from langchain_core.tools import tool
from langchain_groq import ChatGroq

parser_llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0,
                      rate_limiter=groq_rate_limiter)


class RequirementDraft(BaseModel):
    """One extracted requirement, before an ID is assigned."""
    text: str = Field(description="The requirement statement, lightly normalized")
    category: Optional[str] = Field(default=None, description="Optional grouping")


class ParsedRequirementsDraft(BaseModel):
    requirements: list[RequirementDraft] = Field(
        description="Every distinct requirement found, in the order they appear"
    )


structured_parser_llm = parser_llm.with_structured_output(ParsedRequirementsDraft)


@tool
def parse_requirements(document_text: str) -> list[dict]:
    """Parse a raw requirements document into structured Requirement items with stable IDs.

    Use this whenever you have unstructured requirements text and need it converted into
    individually addressable records for review.
    """
    if not document_text or not document_text.strip():
        return []

    prompt = f"""Extract every distinct requirement statement from the document below.
A requirement is a single sentence describing something the system shall/should/must do.
Split compound sentences into separate requirements if they describe separate behaviors.
Ignore headings, examples, and non-requirement prose.

DOCUMENT:
{document_text}
"""
    draft = structured_parser_llm.invoke(prompt)
    requirements = [
        Requirement(id=f"REQ-{i+1:03d}", text=d.text.strip(), category=d.category)
        for i, d in enumerate(draft.requirements)
    ]
    print(f"  [parser] extracted {len(requirements)} requirement(s) from {len(document_text)} chars")
    return [r.model_dump() for r in requirements]


def load_document_text(file_path: str) -> str:
    """Read raw text from a .txt or .docx file."""
    if file_path.endswith(".docx"):
        from docx import Document
        return "\n".join(p.text for p in Document(file_path).paragraphs)
    with open(file_path, "r", encoding="utf-8") as f:
        return f.read()


print("Parser tool ready.")

Parser tool ready.


## 5. Knowledge base and RAG pipeline

Four reference documents — quality criteria, the requirement-writing template and its
prohibited-terms list, a sample SRS exemplar, and a catalogue of conflict patterns. They are
our own summaries of established requirements-engineering guidance, so the repository
carries no licensing problem.

**Pipeline:** load → split → embed → store → retrieve, each step explicit.

**RAG choice: Hybrid.** Retrieval inside the reviewers is mandatory and deterministic
(2-Step), while `search_requirements_standards` is a tool the Supervisor can call on demand
(Agentic). Pure Agentic was rejected because a reviewer that skips retrieval falls back on
its own taste, which is exactly the subjective judgement the Standards Reviewer exists to
replace. Pure 2-Step was rejected because ad-hoc lookups during synthesis are unpredictable
in topic and number. The cost is one guaranteed retrieval per reviewer pass — milliseconds
against an in-process store, in exchange for guaranteed grounding.

In [7]:
# Writes the reference corpus to disk so the loader has real documents to read.
import os, textwrap
os.makedirs("/content/rag_corpus", exist_ok=True)

CORPUS = {}

CORPUS["conflict_patterns.md"] = r"""
# Common Requirement Conflict Patterns (review reference)

A conflict exists when two requirements cannot both be satisfied by any single
implementation. Reviewers should test each candidate pair against these patterns.

## Pattern 1 — Permission versus prohibition
One requirement grants an actor an unrestricted ability while another forbids the same
actor the same ability under a condition that can occur. Signal words: "at any time",
"always", "never", "under no circumstances".
Example: "Users may edit their profile at any time" conflicts with "Users cannot edit
their profile after account verification", because a verified user falls under both.

## Pattern 2 — Contradictory quantitative limits
Two requirements state different values for the same measurable property under the same
conditions, such as two different response-time ceilings or two different retention
periods for the same record type.

## Pattern 3 — Incompatible ordering
Two requirements each demand that a different step occur first in the same workflow, for
example requiring payment before confirmation while also requiring confirmation before
payment.

## Pattern 4 — Mutually exclusive states
Two requirements demand that the same entity be in two states that cannot hold at once,
such as requiring that a record be permanently immutable while also requiring that it be
editable by an administrator.

## Pattern 5 — Deadline versus exception without precedence
A general deadline rule and a narrower exception rule both apply to the same action, and
neither states which takes precedence. This is a genuine conflict until an explicit
precedence rule is added.

## What is NOT a conflict
Two requirements that address different actors, different entities, or mutually exclusive
preconditions are not in conflict. A general rule followed by an exception that explicitly
names its precedence is not a conflict. Requirements at different levels of detail, where
one refines the other, are not in conflict.
"""

CORPUS["re_quality_criteria.md"] = r"""
# Requirements Quality Criteria (study extract)

A well-written requirement is expected to satisfy several quality characteristics.
These characteristics are drawn from established requirements-engineering guidance
and restated here in our own words for use as a review reference.

## Unambiguous
A requirement is unambiguous when it has exactly one possible interpretation.
Words such as "quickly", "fast", "user-friendly", "efficient", "robust", "flexible",
"as appropriate", "if necessary", "etc." and "and/or" introduce ambiguity because two
readers can reasonably disagree about what satisfies them. Replace these with a stated
value, a stated condition, or a named standard.

## Verifiable (testable)
A requirement is verifiable when a finite, cost-effective process exists to check that
the delivered system meets it. In practice this means the requirement must have an
objective pass/fail criterion. A requirement that cannot be verified by inspection,
analysis, demonstration, or test is not verifiable and should be rewritten.
Non-verifiable phrasing includes "the system shall be easy to use", "the system shall
work well", and "the system shall be secure" with no stated threshold or standard.

## Measurable performance
Performance requirements must state a numeric target, the unit of measurement, and the
conditions under which the target applies. The recommended form is:
"The system shall <action> within <number> <unit> under <stated load or condition>."
For example, a response-time requirement should state both the time limit and the
concurrent-user load at which that limit must hold.

## Complete
A requirement is complete when it states the triggering condition, the actor, the system
response, and the outcome, with no "to be determined" content remaining. A requirements
set is complete when it covers every capability described in the project scope, including
error handling and exception behaviour for each described capability.

## Consistent
A requirements set is consistent when no requirement contradicts another. Conflicts
commonly appear as a general permission paired with a narrower prohibition covering the
same actor and action, as two different values specified for the same quantity, or as two
requirements that impose incompatible ordering on the same operation.

## Singular
A requirement should state exactly one need. A sentence joining two behaviours with "and"
usually should be split into two requirements so that each can be verified separately.

## Feasible and traceable
A requirement must be achievable within known constraints, and must be traceable back to a
stated stakeholder need or scope item and forward to its verification method.
"""

CORPUS["sample_srs_registration.md"] = r"""
# Sample SRS Extract — University Course Registration (reference exemplar)

This extract is provided as a model of correctly written requirements for a course
registration domain. Reviewers may compare submitted requirements against these.

REQ-101 The system shall allow an authenticated student to register for a course section
when the section has at least one available seat and the student has no outstanding
registration hold.
Acceptance: Given a section with 1 seat free and a student with no hold, when the student
submits a registration request, then the seat count decreases by 1 and the student appears
on the section roster.

REQ-102 The system shall prevent registration for a course section whose enrolled count
equals its seat capacity, and shall return an explanatory message naming the full section.

REQ-103 The system shall allow a student to drop a course section up to and including the
published add/drop deadline for the current term.

REQ-104 The system shall reject any drop request submitted after the published add/drop
deadline and shall return an explanatory message stating the deadline date.

REQ-105 The system shall notify a waitlisted student within 5 minutes of a seat becoming
available in the section for which they are waitlisted.

REQ-106 The system shall allow an academic advisor to view the complete registration
history of any student assigned to that advisor.

REQ-107 The system shall allow an academic advisor to override a registration hold, and
shall record the advisor identity, timestamp, and stated reason for every override.

REQ-108 The system shall generate a weekly enrollment report listing current enrolled
count and seat capacity for every course section, and shall make it available to
department administrators each Monday by 06:00 local time.

REQ-109 The system shall complete a registration transaction within 3 seconds under a load
of 500 concurrent registration requests.

REQ-110 The system shall record an audit entry for every registration, drop, and override
action, containing actor identity, action type, target section, and timestamp.
"""

CORPUS["srs_writing_template.md"] = r"""
# Requirement Statement Template and House Style

## Mandatory sentence form
Every functional requirement in this organisation is written as:

  The system shall <observable behaviour> [when <trigger>] [within <measurable limit>].

The auxiliary verb "shall" denotes a binding requirement. "Should" denotes a
recommendation and must not be used for contractual requirements. "Will" denotes a
statement of fact about the environment, not a requirement on the system.

## Identifier convention
Each requirement carries a unique, stable identifier of the form REQ-NNN. Identifiers are
never reused after a requirement is deleted.

## Acceptance criteria
Every requirement is accompanied by at least one acceptance criterion stated in
Given / When / Then form, so that a tester can determine pass or fail without consulting
the author.

## Prohibited vague terms
The following terms are rejected in review unless immediately followed by a numeric
threshold or a named external standard: quickly, fast, slow, user-friendly, intuitive,
easy, simple, seamless, efficient, optimised, robust, reliable, scalable, secure,
appropriate, adequate, sufficient, minimal, maximal, state-of-the-art, modern.

## Performance requirement examples
Acceptable: The system shall return search results within 2 seconds for result sets of up
to 500 records, with 200 concurrent users.
Rejected: The system shall return search results quickly.

## Availability requirements
Availability is expressed as a percentage measured over a stated period, together with the
maximum permitted duration of a single unplanned outage.
Acceptable: The system shall maintain 99.5% availability measured monthly, with no single
unplanned outage exceeding 30 minutes.
Rejected: The system shall be highly available.
"""


for name, body in CORPUS.items():
    with open(f"/content/rag_corpus/{name}", "w", encoding="utf-8") as fh:
        fh.write(body)
    print(f"wrote /content/rag_corpus/{name}  ({len(body)} chars)")

wrote /content/rag_corpus/conflict_patterns.md  (1986 chars)
wrote /content/rag_corpus/re_quality_criteria.md  (2691 chars)
wrote /content/rag_corpus/sample_srs_registration.md  (2108 chars)
wrote /content/rag_corpus/srs_writing_template.md  (1775 chars)


In [8]:
from pathlib import Path
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_huggingface import HuggingFaceEmbeddings

CORPUS_DIR = "/content/rag_corpus"

# 1. LOAD
standards_docs = [
    Document(page_content=p.read_text(encoding="utf-8"), metadata={"source": str(p)})
    for p in sorted(Path(CORPUS_DIR).glob("**/*.md"))
]
assert standards_docs, f"No documents in {CORPUS_DIR} — run the corpus cell first."
print(f"[1/5] loaded {len(standards_docs)} document(s)")
for d in standards_docs:
    print(f"      {Path(d.metadata['source']).name}  ({len(d.page_content)} chars)")

# 2. SPLIT — heading separators first, so a chunk rarely straddles two rules
splitter = RecursiveCharacterTextSplitter(
    chunk_size=800, chunk_overlap=120,
    separators=["\n## ", "\n### ", "\n\n", "\n", " ", ""],
)
standards_chunks = splitter.split_documents(standards_docs)
print(f"[2/5] split into {len(standards_chunks)} chunks "
      f"(avg {sum(len(c.page_content) for c in standards_chunks)//len(standards_chunks)} chars)")

# 3. EMBED — runs locally, no API key
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
print(f"[3/5] embedding model loaded (dim={len(embeddings.embed_query('probe'))})")

# 4. STORE
standards_store = InMemoryVectorStore.from_documents(standards_chunks, embeddings)
print(f"[4/5] vector store built with {len(standards_chunks)} vectors")

# 5. RETRIEVE
standards_retriever = standards_store.as_retriever(search_kwargs={"k": 4})
print("[5/5] retriever ready (k=4)")


def retrieve_standards(query: str, k: int = 4) -> str:
    """Deterministic retrieval, used as step 1 inside the RAG-grounded reviewers."""
    hits = standards_store.similarity_search(query, k=k)
    if not hits:
        return "(no relevant standards retrieved)"
    return "\n\n---\n\n".join(
        f"[source: {Path(h.metadata['source']).name}]\n{h.page_content}" for h in hits
    )


@tool
def search_requirements_standards(query: str) -> str:
    """Search the requirements-engineering knowledge base for guidance relevant to a query.

    Contains quality criteria, the requirement-writing template and prohibited vague terms,
    a sample SRS exemplar, and catalogued conflict patterns. Use this to check how a
    requirement should be written, whether a term is disallowed, or what a correctly
    written equivalent looks like.
    """
    return retrieve_standards(query, k=4)

[1/5] loaded 4 document(s)
      conflict_patterns.md  (1986 chars)
      re_quality_criteria.md  (2691 chars)
      sample_srs_registration.md  (2108 chars)
      srs_writing_template.md  (1775 chars)
[2/5] split into 13 chunks (avg 656 chars)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

[3/5] embedding model loaded (dim=384)
[4/5] vector store built with 13 vectors
[5/5] retriever ready (k=4)


## 6. Reviewer agents

Four specialists. Each returns typed `ReviewFinding` records via `with_structured_output`,
never free text parsed by code. The two RAG-grounded reviewers retrieve first, then make a
single grounded generation call.

In [9]:
# Completeness and Ambiguity reviewers.
reviewer_llm_b = ChatGroq(model="llama-3.3-70b-versatile", temperature=0,
                          rate_limiter=groq_rate_limiter)
from langgraph.func import task


class MissingRequirementDraft(BaseModel):
    """One capability described in the project but not covered by any requirement."""
    reason: str = Field(description="Why this is missing, referencing the project description")
    suggested_change: str = Field(description="The requirement to add, as 'The system shall ...'")
    severity: Literal["low", "medium", "high", "critical"]


class CompletenessFindings(BaseModel):
    gaps: list[MissingRequirementDraft] = Field(
        description="One per described capability with no matching requirement; empty if fully covered"
    )


completeness_llm = reviewer_llm_b.with_structured_output(CompletenessFindings)


@task(retry_policy=REVIEWER_RETRY)
def completeness_reviewer(project_description: Optional[str],
                          requirements: list[Requirement]) -> list[ReviewFinding]:
    """Finds capabilities the project description promises but no requirement covers."""
    if not project_description:
        return []

    existing = "\n".join(f"- {r.id}: {r.text}" for r in requirements) or "(none provided)"
    prompt = f"""You are reviewing a requirements document for completeness against its project description.

PROJECT DESCRIPTION:
{project_description}

EXISTING REQUIREMENTS:
{existing}

Find every capability mentioned or clearly implied in the project description that has
NO corresponding requirement above. Do not flag anything already covered, even if worded
differently. If everything is covered, return an empty list.
"""
    result = completeness_llm.invoke(prompt)
    findings = [
        ReviewFinding(requirement_id=f"GAP-{i+1:03d}", issue_type="missing",
                      severity=g.severity, reason=g.reason, suggested_change=g.suggested_change)
        for i, g in enumerate(result.gaps)
    ]
    print(f"  [completeness] found {len(findings)} gap(s)")
    return findings


class AmbiguityFindingDraft(BaseModel):
    """One requirement flagged for vague or subjective wording."""
    requirement_id: str = Field(description="ID of the flagged requirement, copied exactly")
    reason: str = Field(description="Which terms are vague and why they can't be verified")
    suggested_change: str = Field(description="Rewritten with a measurable threshold")
    severity: Literal["low", "medium", "high", "critical"]


class AmbiguityFindings(BaseModel):
    findings: list[AmbiguityFindingDraft] = Field(
        description="One per requirement with vague wording; empty list if none"
    )


ambiguity_llm = reviewer_llm_b.with_structured_output(AmbiguityFindings)


@task(retry_policy=REVIEWER_RETRY)
def ambiguity_reviewer(requirements: list[Requirement]) -> list[ReviewFinding]:
    """Flags vague or subjective terms and proposes a measurable rewrite."""
    if not requirements:
        return []

    listed = "\n".join(f"- {r.id}: {r.text}" for r in requirements)
    prompt = f"""Review these requirements for vague or subjective wording that cannot be
objectively tested (e.g. "quickly", "user-friendly", "fast", "easy", "robust", "efficient").

REQUIREMENTS:
{listed}

For each requirement containing such wording, explain what is vague and rewrite it with a
concrete, measurable threshold. If a requirement is already specific and testable, do NOT
include it. An empty findings list is a valid and expected answer.
"""
    result = ambiguity_llm.invoke(prompt)
    findings = [
        ReviewFinding(requirement_id=f.requirement_id, issue_type="ambiguous",
                      severity=f.severity, reason=f.reason, suggested_change=f.suggested_change)
        for f in result.findings
    ]
    findings = keep_known_requirement_ids(findings, requirements, "ambiguity")
    print(f"  [ambiguity] found {len(findings)} vague requirement(s)")
    return findings

In [10]:
# Conflict and Testability & Standards reviewers — both RAG-grounded (2-Step).
reviewer_llm_c = ChatGroq(model="llama-3.3-70b-versatile", temperature=0,
                          rate_limiter=groq_rate_limiter)


class ConflictDraft(BaseModel):
    """One contradiction found between two requirements."""
    requirement_id: str = Field(description="ID of the FIRST requirement in the pair, copied exactly")
    conflicting_with_id: str = Field(description="ID of the SECOND requirement in the pair, copied exactly")
    pattern: str = Field(description="Which conflict pattern from the retrieved reference this matches")
    reason: str = Field(description="Why both cannot be satisfied at once, naming both IDs")
    suggested_change: str = Field(description="A rewrite that removes the contradiction")
    severity: Literal["low", "medium", "high", "critical"]


class ConflictFindings(BaseModel):
    conflicts: list[ConflictDraft] = Field(
        description="One per genuinely contradictory pair; empty if mutually consistent"
    )


conflict_llm = reviewer_llm_c.with_structured_output(ConflictFindings)


@task(retry_policy=REVIEWER_RETRY)
def conflict_reviewer(requirements: list[Requirement]) -> list[ReviewFinding]:
    """Detects contradictions between requirements, grounded in the retrieved conflict patterns."""
    if len(requirements) < 2:
        return []

    # Step 1 — retrieve
    context = retrieve_standards(
        "requirement conflict patterns contradiction permission prohibition "
        "incompatible ordering mutually exclusive what is not a conflict", k=3)

    # Step 2 — generate, grounded
    listed = "\n".join(f"- {r.id}: {r.text}" for r in requirements)
    prompt = f"""You are the Conflict Reviewer. Find pairs of requirements that contradict
each other, using the reference material below to decide what does and does not count.

REFERENCE MATERIAL:
{context}

REQUIREMENTS UNDER REVIEW:
{listed}

Report a conflict only when two requirements cannot both be satisfied by any single
implementation. Requirements about different actors, different entities, or mutually
exclusive preconditions are NOT conflicts. A general rule with an exception that explicitly
states its precedence is NOT a conflict.

Copy requirement IDs exactly. Report ONLY on requirements listed under REQUIREMENTS UNDER
REVIEW — the reference material has its own example IDs, which are illustrations only.
If nothing genuinely contradicts, return an empty list.
"""
    result = conflict_llm.invoke(prompt)
    findings = [
        ReviewFinding(
            requirement_id=c.requirement_id,
            issue_type="conflicting",
            severity=c.severity,
            # ReviewFinding carries one id, so the partner is named in the reason
            # rather than changing the shared schema.
            reason=f"Conflicts with {c.conflicting_with_id} [{c.pattern}]. {c.reason}",
            suggested_change=c.suggested_change,
        )
        for c in result.conflicts
    ]
    findings = keep_known_requirement_ids(findings, requirements, "conflict")
    print(f"  [conflict] found {len(findings)} contradiction(s)")
    return findings


class TestabilityDraft(BaseModel):
    """One requirement that cannot be objectively verified, or breaches house style."""
    requirement_id: str = Field(description="ID of the flagged requirement, copied exactly")
    reason: str = Field(description="Why it has no pass/fail criterion, or which rule it breaches")
    suggested_change: str = Field(description="Rewritten so a tester can determine pass/fail")
    severity: Literal["low", "medium", "high", "critical"]


class TestabilityFindings(BaseModel):
    findings: list[TestabilityDraft] = Field(
        description="One per unverifiable or non-compliant requirement; empty if all are fine"
    )


testability_llm = reviewer_llm_c.with_structured_output(TestabilityFindings)


@task(retry_policy=REVIEWER_RETRY)
def testability_standards_reviewer(requirements: list[Requirement]) -> list[ReviewFinding]:
    """Flags requirements with no objective pass/fail criterion, grounded in retrieved standards."""
    if not requirements:
        return []

    # Step 1 — retrieve, querying with the requirement text so passages match THIS document
    query = ("verifiable testable acceptance criteria measurable threshold pass fail "
             "prohibited vague terms requirement sentence form "
             + " ".join(r.text for r in requirements))
    context = retrieve_standards(query, k=4)

    # Step 2 — generate, grounded
    listed = "\n".join(f"- {r.id}: {r.text}" for r in requirements)
    prompt = f"""You are the Testability & Standards Reviewer. Judge each requirement against
the retrieved standards below, not against your own preferences.

RETRIEVED STANDARDS:
{context}

REQUIREMENTS UNDER REVIEW:
{listed}

Flag a requirement when EITHER:
(a) it has no objective pass/fail criterion, or
(b) it breaches a rule in the retrieved standards, such as a prohibited vague term with no
    numeric threshold.

Cite the specific rule in your reason and rewrite it following the retrieved template.
Do not flag a requirement that is already specific and verifiable, and do not flag one
solely for missing acceptance criteria when its behaviour is already objectively observable.

Report ONLY on requirements listed under REQUIREMENTS UNDER REVIEW — the retrieved standards
contain their own example IDs (REQ-1nn), which are illustrations only.
An empty findings list is a valid and expected answer.
"""
    result = testability_llm.invoke(prompt)
    findings = [
        ReviewFinding(requirement_id=f.requirement_id, issue_type="untestable",
                      severity=f.severity, reason=f.reason, suggested_change=f.suggested_change)
        for f in result.findings
    ]
    findings = keep_known_requirement_ids(findings, requirements, "testability")
    print(f"  [testability] found {len(findings)} unverifiable requirement(s)")
    return findings

## 7. Supervisor

Two roles, one agent. As **router** it makes an LLM call with constrained output to decide
which reviewers an input needs — not a keyword rule. As **synthesizer** it merges the
selected reviewers' findings, removes duplicates, and orders by severity.

A deterministic guardrail sits *under* the LLM decision for cases that are not a judgement
call but a logical impossibility: completeness review needs a description to compare
against, conflict review needs at least two requirements.

In [11]:
llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0,
               rate_limiter=groq_rate_limiter)


class RouterDecision(BaseModel):
    """Which reviewers should run on this input."""
    run_completeness: bool = Field(description="True only if a project description exists")
    run_ambiguity: bool = Field(description="True if at least one requirement exists")
    run_conflict: bool = Field(description="True only if at least two requirements exist")
    run_testability: bool = Field(description="True if at least one requirement exists")
    reason: str = Field(description="One sentence justifying the selection")


router_llm = llm.with_structured_output(RouterDecision)


def _enforce_hard_guardrails(decision, project_description, requirements):
    """Deterministic floor under the LLM's choice — blocks structurally impossible work only."""
    if not project_description:
        decision.run_completeness = False
    if len(requirements) < 2:
        decision.run_conflict = False
    if not requirements:
        decision.run_ambiguity = False
        decision.run_testability = False
    return decision


@task(retry_policy=REVIEWER_RETRY)
def supervisor_route(project_description: Optional[str],
                     requirements: list[Requirement]) -> RouterDecision:
    """Supervisor-as-router: the LLM decides who runs, via constrained structured output."""
    listed = ("\n".join(f"- [{r.id}] {r.text}" for r in requirements)) if requirements else "(none)"
    prompt = f"""You are the Supervisor for a requirements-review pipeline.
Decide which reviewers should run on this input. Do not run a reviewer whose job is
logically impossible given the input (e.g. completeness with no project description).

Output strict JSON booleans (true/false) for the routing fields, not strings.

Project description: {project_description if project_description else "(none provided)"}
Number of requirements: {len(requirements)}
Requirements:
{listed}
"""
    decision = router_llm.invoke(prompt)
    return _enforce_hard_guardrails(decision, project_description, requirements)


_SEVERITY_RANK = {"critical": 0, "high": 1, "medium": 2, "low": 3}


@task
def supervisor_synthesize(finding_lists: list[list[ReviewFinding]]) -> list[ReviewFinding]:
    """Flatten, dedupe by (requirement_id, issue_type) keeping the worst, sort by severity."""
    flattened = [f for group in finding_lists for f in group]
    deduped: dict = {}
    for f in flattened:
        key = (f.requirement_id, f.issue_type)
        current = deduped.get(key)
        if current is None or _SEVERITY_RANK[f.severity] < _SEVERITY_RANK[current.severity]:
            deduped[key] = f
    return sorted(deduped.values(), key=lambda f: _SEVERITY_RANK[f.severity])


def render_report(findings: list[ReviewFinding]) -> str:
    """Severity-ordered report with a per-severity summary line."""
    if not findings:
        return "REQUIREMENTS REVIEW REPORT\n\nNo findings — the document passed every check that ran."

    counts = {s: sum(1 for f in findings if f.severity == s) for s in _SEVERITY_RANK}
    summary = "  ".join(f"{s}:{counts[s]}" for s in ("critical", "high", "medium", "low"))

    lines = ["REQUIREMENTS REVIEW REPORT", "",
             f"{len(findings)} finding(s)    {summary}", ""]
    for f in findings:
        lines.append(f"[{f.severity.upper()}] {f.issue_type.upper()} — {f.requirement_id}")
        lines.append(f"   why : {wrap(f.reason, 9)}")
        lines.append(f"   fix : {wrap(f.suggested_change, 9)}")
        lines.append("")
    return "\n".join(lines)


print("Supervisor ready: router + synthesizer.")

Supervisor ready: router + synthesizer.


## 8. Memory

**Short-term** is the checkpointer, keyed by `thread_id`: it carries state across one review
session and is what lets an interrupted run resume.

**Long-term** is a separate `InMemoryStore`, keyed by `project_id`. Product Owner decisions
written in one thread are readable from a different thread in a later session — that
independence from `thread_id` is what makes it long-term rather than a growing message list.

In [12]:
from uuid import uuid4
from datetime import datetime, timezone
from langgraph.store.memory import InMemoryStore
from langgraph.func import entrypoint
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import interrupt, Command

checkpointer = InMemorySaver()          # short-term: per-thread state
project_memory_store = InMemoryStore()  # long-term: per-project facts


def _project_memory_namespace(project_id: str):
    project_id = project_id.strip()
    if not project_id:
        raise ValueError("project_id must not be empty.")
    return (project_id, "po_decisions")


@tool
def save_project_memory(project_id: str, requirement_id: str, decision: str,
                        final_text: str = "", notes: str = "") -> str:
    """Save a Product Owner decision to long-term project memory. Returns the memory id."""
    memory_id = str(uuid4())
    project_memory_store.put(
        _project_memory_namespace(project_id),
        memory_id,
        {
            "project_id": project_id,
            "requirement_id": requirement_id,
            "decision": decision,
            "final_text": final_text or None,
            "notes": notes or None,
            "saved_at": datetime.now(timezone.utc).isoformat(),
        },
    )
    return memory_id


@tool
def retrieve_project_memory(project_id: str) -> list[dict]:
    """Retrieve every stored Product Owner decision for a project, from any thread."""
    items = project_memory_store.search(_project_memory_namespace(project_id), limit=100)
    return [{"memory_id": item.key, **item.value} for item in items]


print("Memory ready: checkpointer (short-term) + project store (long-term).")

Memory ready: checkpointer (short-term) + project store (long-term).


## 9. Human-in-the-loop

The pipeline pauses with `interrupt()` before anything is written, presents every finding to
the Product Owner, and resumes on `Command(resume=...)`. Each finding gets approve, edit, or
reject; decisions are then translated into `ApprovedChange` records and persisted to
long-term memory.

In [13]:
class POFindingDecision(BaseModel):
    finding_index: int = Field(ge=0)
    decision: Literal["approve", "edit", "reject"]
    edited_text: Optional[str] = None
    notes: Optional[str] = None


class POReviewResponse(BaseModel):
    decisions: list[POFindingDecision]


@task
def requirements_approval_hitl(project_id: str, findings: list[ReviewFinding],
                               previous_project_decisions: list[dict]) -> dict:
    """Pause the run and hand every finding to the Product Owner."""
    return interrupt({
        "type": "requirements_review_approval",
        "project_id": project_id,
        "message": "Review every finding and choose approve, edit, or reject.",
        "previous_project_decisions": previous_project_decisions,
        "findings": [
            {"finding_index": i, **f.model_dump(),
             "allowed_decisions": ["approve", "edit", "reject"]}
            for i, f in enumerate(findings)
        ],
    })


@task
def build_approved_changes(findings: list[ReviewFinding], po_response: dict) -> list[ApprovedChange]:
    """Translate PO decisions into Editor instructions. Every finding needs exactly one."""
    response = POReviewResponse(**po_response)

    by_index = {}
    for d in response.decisions:
        if d.finding_index in by_index:
            raise ValueError(f"Duplicate decision for finding {d.finding_index}.")
        by_index[d.finding_index] = d
    if set(by_index) != set(range(len(findings))):
        raise ValueError("Every finding must have exactly one Product Owner decision.")

    changes = []
    for index, finding in enumerate(findings):
        d = by_index[index]

        if d.decision == "approve":
            action = "add" if finding.issue_type == "missing" else "replace"
            changes.append(ApprovedChange(requirement_id=finding.requirement_id, action=action,
                                          edited_text=finding.suggested_change, notes=d.notes))

        elif d.decision == "edit":
            if not d.edited_text or not d.edited_text.strip():
                raise ValueError(f"Finding {index} requires edited_text.")
            # A 'missing' finding has no existing requirement to edit in place — the PO
            # editing it still means ADD. Emitting apply_po_edit would hand the Editor a
            # GAP-nnn id that does not exist.
            action = "add" if finding.issue_type == "missing" else "apply_po_edit"
            changes.append(ApprovedChange(requirement_id=finding.requirement_id, action=action,
                                          edited_text=d.edited_text.strip(), notes=d.notes))

        else:
            changes.append(ApprovedChange(requirement_id=finding.requirement_id,
                                          action="leave_unchanged", edited_text=None, notes=d.notes))

    return changes


@task
def persist_po_decisions(project_id: str, findings: list[ReviewFinding],
                         approved_changes: list[ApprovedChange]) -> list[str]:
    """Write each decision to long-term memory so later sessions can see it."""
    if len(findings) != len(approved_changes):
        raise ValueError("findings and approved_changes must be the same length.")

    _decision_for = {"leave_unchanged": "reject", "apply_po_edit": "edit"}
    memory_ids = []
    for finding, change in zip(findings, approved_changes):
        memory_ids.append(save_project_memory.invoke({
            "project_id": project_id,
            "requirement_id": finding.requirement_id,
            "decision": _decision_for.get(change.action, "approve"),
            "final_text": change.edited_text or "",
            "notes": change.notes or "",
        }))
    return memory_ids


def scripted_po_decision(item: dict) -> dict:
    """Default Product Owner answers, rotating approve / edit / reject.

    Used so the notebook runs unattended: input() would block "Run all" forever and leave
    no captured output. The pause and the resume are real either way — only the source of
    the answers changes. See the interactive alternative in the demo section.
    """
    index = item["finding_index"]
    if index % 3 == 0:
        return {"finding_index": index, "decision": "approve", "edited_text": None,
                "notes": "Accepted the reviewer's suggested wording."}
    if index % 3 == 1:
        return {"finding_index": index, "decision": "edit",
                "edited_text": f"{item['suggested_change']} Verified against the team glossary.",
                "notes": "Approved with a Product Owner wording change."}
    return {"finding_index": index, "decision": "reject", "edited_text": None,
            "notes": "Deferred — out of scope for this release."}


def interactive_po_decision(item: dict) -> dict:
    """Prompt the Product Owner at the console. Only used when explicitly selected."""
    while True:
        choice = input(f"\nFinding {item['finding_index']} — approve / edit / reject: ").strip().lower()
        if choice in {"approve", "edit", "reject"}:
            break
        print("Please enter approve, edit, or reject.")

    edited_text = None
    if choice == "edit":
        edited_text = input("Enter the edited requirement: ").strip()
        while not edited_text:
            edited_text = input("Cannot be empty. Enter the edited requirement: ").strip()

    notes = input("Optional notes (Enter to skip): ").strip()
    return {"finding_index": item["finding_index"], "decision": choice,
            "edited_text": edited_text, "notes": notes or None}


print("Human-in-the-loop ready: interrupt -> decisions -> ApprovedChange -> memory.")

Human-in-the-loop ready: interrupt -> decisions -> ApprovedChange -> memory.


## 10. Requirements Editor

Applies approved changes and nothing else. It makes no judgement about whether a suggestion
was correct — that decision was the Product Owner's, one step earlier.

In [14]:
class EditorError(Exception):
    """Raised when an ApprovedChange cannot be applied."""


def _apply_one(by_id: dict, change: ApprovedChange, next_new_id: int) -> int:
    if change.action == "leave_unchanged":
        return next_new_id

    if change.action in ("replace", "apply_po_edit"):
        if change.requirement_id not in by_id:
            raise EditorError(f"Cannot {change.action} unknown requirement_id={change.requirement_id!r}")
        if not change.edited_text:
            raise EditorError(f"{change.action} on {change.requirement_id!r} missing edited_text")
        by_id[change.requirement_id].text = change.edited_text
        return next_new_id

    if change.action == "add":
        if not change.edited_text:
            raise EditorError("add action missing edited_text")
        new_id = change.requirement_id
        # A completeness finding arrives as GAP-nnn, which is a finding id, not a
        # requirement id. Anything missing, taken, or not REQ-shaped gets a fresh id.
        if not new_id or new_id in by_id or not new_id.startswith("REQ-"):
            new_id = f"REQ-NEW-{next_new_id:03d}"
            next_new_id += 1
        by_id[new_id] = Requirement(id=new_id, text=change.edited_text)
        return next_new_id

    raise EditorError(f"Unknown action: {change.action!r}")


def _apply_approved_changes(requirements: list[Requirement],
                            decisions: list[ApprovedChange]) -> list[Requirement]:
    """Apply a batch of approved changes and return the updated requirements list."""
    by_id = {r.id: r for r in requirements}
    next_new_id = 1
    for change in decisions:
        next_new_id = _apply_one(by_id, change, next_new_id)
    return list(by_id.values())


apply_approved_changes = task(_apply_approved_changes)

print("Editor ready.")

Editor ready.


## 11. Pipelines

**Workflow pattern: Orchestrator–Worker.** The Supervisor is the orchestrator; the four
reviewers are workers. It fits because the reviewers are independent specialists over the
same input, and which of them are needed is decided per input rather than fixed in advance.
Futures are collected before any `.result()` so selected reviewers run concurrently.

Two entrypoints: `review_pipeline` reviews and reports; `requirements_copilot` is the full
system, adding memory, the approval pause, and the Editor.

In [15]:
@entrypoint(checkpointer=checkpointer)
def review_pipeline(inputs: dict) -> dict:
    """Review only: route -> selected reviewers -> merged report."""
    project_description = inputs.get("project_description")
    requirements = [Requirement(**r) for r in inputs.get("requirements", [])]

    decision = supervisor_route(project_description, requirements).result()
    print(f"  [router] {decision.reason}")

    futures = []
    if decision.run_completeness:
        futures.append(completeness_reviewer(project_description, requirements))
    if decision.run_ambiguity:
        futures.append(ambiguity_reviewer(requirements))
    if decision.run_conflict:
        futures.append(conflict_reviewer(requirements))
    if decision.run_testability:
        futures.append(testability_standards_reviewer(requirements))

    findings = supervisor_synthesize([f.result() for f in futures]).result()

    return {
        "routing_decision": decision.model_dump(),
        "findings": [f.model_dump() for f in findings],
        "report": render_report(findings),
    }


@entrypoint(checkpointer=checkpointer)
def requirements_copilot(inputs: dict) -> dict:
    """Full system: memory -> route -> review -> approval pause -> Editor -> updated document."""
    project_id = inputs["project_id"]
    project_description = inputs.get("project_description")
    requirements = [Requirement(**r) for r in inputs.get("requirements", [])]

    previous_decisions = retrieve_project_memory.invoke({"project_id": project_id})

    decision = supervisor_route(project_description, requirements).result()
    print(f"  [router] {decision.reason}")

    futures = []
    if decision.run_completeness:
        futures.append(completeness_reviewer(project_description, requirements))
    if decision.run_ambiguity:
        futures.append(ambiguity_reviewer(requirements))
    if decision.run_conflict:
        futures.append(conflict_reviewer(requirements))
    if decision.run_testability:
        futures.append(testability_standards_reviewer(requirements))

    findings = supervisor_synthesize([f.result() for f in futures]).result()

    # Pause here — nothing below this line runs until the Product Owner responds.
    po_response = requirements_approval_hitl(project_id, findings, previous_decisions).result()

    approved_changes = build_approved_changes(findings, po_response).result()
    memory_ids = persist_po_decisions(project_id, findings, approved_changes).result()
    updated = apply_approved_changes(requirements, approved_changes).result()

    return {
        "project_id": project_id,
        "routing_decision": decision.model_dump(),
        "previous_project_decisions": previous_decisions,
        "findings": [f.model_dump() for f in findings],
        "report": render_report(findings),
        "approved_changes": [c.model_dump() for c in approved_changes],
        "saved_memory_ids": memory_ids,
        "updated_requirements": [r.model_dump() for r in updated],
    }


print("Pipelines compiled: review_pipeline, requirements_copilot")

Pipelines compiled: review_pipeline, requirements_copilot


---
# 12. Demos

| Demo | Shows |
|---|---|
| 12.1 | Sample input document + parser |
| 12.2 | RAG retrieval returns real grounded passages |
| 12.3 | Routing is selective — different inputs, different reviewers |
| 12.4 | Full review of a flawed document |
| 12.5 | Approval pause, resume, and the updated document |
| 12.6 | Long-term memory readable from a different thread |
| 12.7 | Error handling: retry attached, fallback degrades safely |
| 12.8 | Observability: tracing on, per-stage latency |

### 12.1 — Sample document and parser

In [16]:
# The demo inputs, written to disk so the notebook is self-contained (Colab clears
# /content on restart). The same two files are committed to the repo under demo_data/.
# The requirements document contains, deliberately: vague terms (REQ-004, REQ-007),
# a contradictory pair (REQ-005 vs REQ-006), and two capabilities the project
# description promises but no requirement covers.
DEMO_DOC_PATH = "/content/demo_requirements_flawed.txt"
DEMO_DESC_PATH = "/content/demo_project_description.txt"

with open(DEMO_DOC_PATH, "w", encoding="utf-8") as f:
    f.write("""BrightPath Course Registration System — Requirements Document

REQ-001: The system shall allow a student to log in using their university ID and password.

REQ-002: The system shall allow a student to browse a list of available courses, showing course name, instructor, and remaining seats.

REQ-003: The system shall allow a student to register for a course if seats are available.

REQ-004: The system shall respond to course search queries quickly.

REQ-005: The system shall never allow a student to drop a course after the add/drop deadline has passed.

REQ-006: The system shall always allow a student to drop a course within 24 hours of registering, even if the add/drop deadline has already passed.

REQ-007: The system shall be reliable and provide a good user experience at all times.

REQ-008: The system shall allow an academic advisor to view a student's registration history.

REQ-009: The system shall allow an academic advisor to override a registration hold on a student's account.
""")

with open(DEMO_DESC_PATH, "w", encoding="utf-8") as f:
    f.write("""BrightPath Course Registration System — Project Description

BrightPath is a web-based course registration system for university students.
The platform allows students to browse available courses, register for courses,
and drop courses they are no longer able to attend. Because popular courses
often fill up quickly, students should be notified automatically when a seat
becomes available in a course they are waitlisted for. Academic advisors need
to be able to review a student's registration history and override registration
holds when necessary. The system must also generate a weekly enrollment report
for department administrators showing current seat counts per course.
""")

banner("12.1  parser")

PROJECT_DESCRIPTION = load_document_text(DEMO_DESC_PATH)
doc_text = load_document_text(DEMO_DOC_PATH)
parsed = parse_requirements.invoke({"document_text": doc_text})

step("parsed requirements")
for r in parsed:
    kv(r["id"], wrap(r["text"], 12), width=10)
ok(f"{len(parsed)} requirements parsed from {len(doc_text)} characters")


12.1  PARSER
  [parser] extracted 9 requirement(s) from 999 chars

--- parsed requirements ------------------------------------------------------
  REQ-001   The system shall allow a student to log in using their university ID and
            password.
  REQ-002   The system shall allow a student to browse a list of available courses,
            showing course name, instructor, and remaining seats.
  REQ-003   The system shall allow a student to register for a course if seats are
            available.
  REQ-004   The system shall respond to course search queries quickly.
  REQ-005   The system shall never allow a student to drop a course after the add/drop
            deadline has passed.
  REQ-006   The system shall always allow a student to drop a course within 24 hours of
            registering, even if the add/drop deadline has already passed.
  REQ-007   The system shall be reliable and provide a good user experience at all times.
  REQ-008   The system shall allow an academic

### 12.2 — RAG retrieval verification

In [17]:
# The failure this guards against is a retriever that silently returns nothing. Ask a
# question whose answer is verbatim in the corpus and assert it comes back.
banner("12.2  rag retrieval")

probe = "What is the maximum permitted duration of a single unplanned outage?"
hits = standards_store.similarity_search(probe, k=3)

kv("query", probe, width=10)
for i, h in enumerate(hits, 1):
    step(f"hit {i} — {Path(h.metadata['source']).name}")
    print(textwrap.indent(h.page_content[:300].strip(), "  "))

assert "30 minutes" in " ".join(h.page_content for h in hits), (
    "RETRIEVAL FAILED — the answer is verbatim in srs_writing_template.md but was not "
    "retrieved. Restart the runtime and run top to bottom before debugging anything else."
)
ok("verbatim answer retrieved ('no single unplanned outage exceeding 30 minutes')")

step("agentic tool entry point")
print(textwrap.indent(
    search_requirements_standards.invoke(
        {"query": "is the word user-friendly allowed in a requirement"})[:300], "  "))


12.2  RAG RETRIEVAL
  query     What is the maximum permitted duration of a single unplanned outage?

--- hit 1 — srs_writing_template.md ------------------------------------------
  ## Availability requirements
  Availability is expressed as a percentage measured over a stated period, together with the
  maximum permitted duration of a single unplanned outage.
  Acceptable: The system shall maintain 99.5% availability measured monthly, with no single
  unplanned outage exceeding 30 min

--- hit 2 — re_quality_criteria.md -------------------------------------------
  ## Measurable performance
  Performance requirements must state a numeric target, the unit of measurement, and the
  conditions under which the target applies. The recommended form is:
  "The system shall <action> within <number> <unit> under <stated load or condition>."
  For example, a response-time requi

--- hit 3 — sample_srs_registration.md ---------------------------------------
  REQ-102 The system shall prevent r

### 12.3 — Routing selectivity

The point of a router is that it does *not* run everything every time. Two inputs, two
different sets of reviewers.

In [18]:
banner("12.3  routing selectivity")

step("run A — description + 3 requirements")
input_a = {
    "project_description": (
        "A course-registration system for university students. Students should be able to "
        "register for courses, drop courses, and receive notifications about seat availability."
    ),
    "requirements": [
        {"id": "REQ-001", "text": "The system shall let a student register for a course."},
        {"id": "REQ-002", "text": "The system shall respond quickly to registration requests."},
        {"id": "REQ-003", "text": "The system shall never allow a student to drop a course after "
                                  "the add/drop deadline, except that the system shall always "
                                  "allow drops within 24 hours of registration."},
    ],
}
result_a = review_pipeline.invoke(input_a, {"configurable": {"thread_id": "demo-routing-a"}})

step("run B — no description, 1 requirement")
input_b = {
    "project_description": None,
    "requirements": [{"id": "REQ-010", "text": "The system shall be user-friendly."}],
}
result_b = review_pipeline.invoke(input_b, {"configurable": {"thread_id": "demo-routing-b"}})

step("decisions compared")
_fields = ["run_completeness", "run_ambiguity", "run_conflict", "run_testability"]
print(f"  {'reviewer':<20}{'run A':<10}{'run B':<10}")
for _f in _fields:
    print(f"  {_f.replace('run_', ''):<20}"
          f"{str(result_a['routing_decision'][_f]):<10}"
          f"{str(result_b['routing_decision'][_f]):<10}")

assert result_a["routing_decision"]["run_completeness"] is True
assert result_b["routing_decision"]["run_completeness"] is False
assert result_b["routing_decision"]["run_conflict"] is False
ok("routing genuinely differs by input — reviewers are selected, not always-on")


12.3  ROUTING SELECTIVITY

--- run A — description + 3 requirements -------------------------------------
  [router] The input contains a project description and multiple requirements, so completeness, ambiguity, conflict, and testability checks are all applicable.
  [conflict] found 0 contradiction(s)
  [testability] found 3 unverifiable requirement(s)
  [ambiguity] found 1 vague requirement(s)
  [completeness] found 2 gap(s)

--- run B — no description, 1 requirement ------------------------------------
  [router] One requirement exists, but no project description is provided.
  [ambiguity] found 1 vague requirement(s)
  [testability] found 1 unverifiable requirement(s)

--- decisions compared -------------------------------------------------------
  reviewer            run A     run B     
  completeness        True      False     
  ambiguity           True      True      
  conflict            True      False     
  testability         True      True      
[PASS] routing genuinel

### 12.4 — Full review of the flawed document

In [19]:
banner("12.4  full review")

review = review_pipeline.invoke(
    {"project_description": PROJECT_DESCRIPTION, "requirements": parsed},
    {"configurable": {"thread_id": "demo-full-review"}},
)

print()
print(review["report"])


12.4  FULL REVIEW
  [router] The input contains a project description and multiple requirements, so completeness, ambiguity, conflict, and testability checks are all applicable.
  [testability] found 9 unverifiable requirement(s)
  [conflict] found 1 contradiction(s)
  [completeness] found 3 gap(s)
  [ambiguity] found 2 vague requirement(s)

REQUIREMENTS REVIEW REPORT

15 finding(s)    critical:1  high:4  medium:10  low:0

[CRITICAL] UNTESTABLE — REQ-007
   why : Prohibited vague terms 'reliable' and 'good user experience'
   fix : The system shall achieve an uptime of 99.9% and respond to user interactions
         within 2 seconds.

[HIGH] MISSING — GAP-002
   why : The project description mentions that the system must generate a weekly
         enrollment report for department administrators, but there is no
         corresponding requirement.
   fix : The system shall generate a weekly enrollment report for department
         administrators showing current seat counts per course.

### 12.5 — Approval pause, resume, and the updated document

`interrupt()` stops the run before anything is written. `Command(resume=...)` completes it.
Answers come from `scripted_po_decision` so the notebook runs unattended — set
`INTERACTIVE_HITL = True` to answer at the console instead.

In [20]:
INTERACTIVE_HITL = False

banner("12.5  human-in-the-loop")

copilot_input = {
    "project_id": "brightpath-capstone",
    "project_description": PROJECT_DESCRIPTION,
    "requirements": parsed,
}
copilot_cfg = {"configurable": {"thread_id": "demo-copilot-session-1"}}

paused = requirements_copilot.invoke(copilot_input, copilot_cfg)
assert "__interrupt__" in paused, "pipeline did not pause"

payload = paused["__interrupt__"][0].value
step("PAUSED — awaiting Product Owner")
kv("message", payload["message"])
kv("findings awaiting review", len(payload["findings"]))
kv("prior decisions on file", len(payload["previous_project_decisions"]))

step("Product Owner decisions")
decide = interactive_po_decision if INTERACTIVE_HITL else scripted_po_decision
po_decisions = []
for item in payload["findings"]:
    record = decide(item)
    po_decisions.append(record)
    print(f"  {record['decision']:<8} {item['issue_type']:<12} {item['requirement_id']}")

resumed = requirements_copilot.invoke(Command(resume={"decisions": po_decisions}), copilot_cfg)
assert "__interrupt__" not in resumed, "pipeline did not resume"
assert len(resumed["approved_changes"]) == len(resumed["findings"])
ok("interrupt() paused the run and Command(resume=...) completed it")


12.5  HUMAN-IN-THE-LOOP
  [router] The input contains a project description and multiple requirements, so completeness, ambiguity, conflict, and testability checks are all applicable.
  [completeness] found 3 gap(s)
  [ambiguity] found 2 vague requirement(s)
  [testability] found 2 unverifiable requirement(s)


  [conflict] found 1 contradiction(s)

--- PAUSED — awaiting Product Owner ------------------------------------------
  message                 Review every finding and choose approve, edit, or reject.
  findings awaiting review8
  prior decisions on file 0

--- Product Owner decisions --------------------------------------------------
  approve  missing      GAP-002
  edit     ambiguous    REQ-007
  reject   untestable   REQ-007
  approve  missing      GAP-001
  edit     missing      GAP-003
  reject   ambiguous    REQ-004
  approve  conflicting  REQ-005
  edit     untestable   REQ-004
  [router] The input contains a project description and multiple requirements, so completeness, ambiguity, conflict, and testability checks are all applicable.
[PASS] interrupt() paused the run and Command(resume=...) completed it


In [21]:
# The deliverable: the requirements document after the Editor applied approved changes only.
banner("12.5b  updated requirements document")

before = {r["id"]: r["text"] for r in parsed}
updated = [Requirement(**r) for r in resumed["updated_requirements"]]

for r in updated:
    if r.id not in before:
        marker = "ADDED  "
    elif before[r.id] != r.text:
        marker = "REVISED"
    else:
        marker = "       "
    print(f"{marker}  {r.id:<12}{wrap(r.text, 21)}")

added = [r for r in updated if r.id not in before]
revised = [r for r in updated if r.id in before and before[r.id] != r.text]
unchanged = [r for r in updated if r.id in before and before[r.id] == r.text]

step("summary")
kv("added", len(added))
kv("revised", len(revised))
kv("unchanged", len(unchanged))

# Nothing may change without approval; a requirement whose only finding was rejected
# must be untouched. An id can be both rejected and approved, so only purely-rejected
# ids are checked here.
authorised = {c["requirement_id"] for c in resumed["approved_changes"]
              if c["action"] != "leave_unchanged"}
rejected_only = {c["requirement_id"] for c in resumed["approved_changes"]
                 if c["action"] == "leave_unchanged"} - authorised

for r in revised:
    assert r.id in authorised, f"{r.id} was modified without an approved change"
for rid in rejected_only:
    if rid in before:
        assert next(x.text for x in updated if x.id == rid) == before[rid], \
            f"a purely rejected finding still altered {rid}"

ok(f"Editor applied approved changes only ({len(authorised)} authorised, "
   f"{len(rejected_only)} left untouched by rejection)")


12.5B  UPDATED REQUIREMENTS DOCUMENT
         REQ-001     The system shall allow a student to log in using their university ID and
                     password.
         REQ-002     The system shall allow a student to browse a list of available courses,
                     showing course name, instructor, and remaining seats.
         REQ-003     The system shall allow a student to register for a course if seats are
                     available.
REVISED  REQ-004     The system shall respond to course search queries within 2 seconds. Verified
                     against the team glossary.
REVISED  REQ-005     Modify REQ-005 to include an exception for dropping a course within 24 hours
                     of registration
         REQ-006     The system shall always allow a student to drop a course within 24 hours of
                     registering, even if the add/drop deadline has already
                     passed.
REVISED  REQ-007     The system shall have an uptime of 99.9% 

### 12.6 — Long-term memory across threads

Session 1 wrote Product Owner decisions under `brightpath-capstone`. A brand-new thread
reads them back. If this survives a thread change, it is long-term memory; if it did not,
it was short-term state wearing a different name.

In [22]:
banner("12.6  cross-thread memory")

@entrypoint(checkpointer=checkpointer)
def read_project_memory(inputs: dict) -> dict:
    return {"memories": retrieve_project_memory.invoke({"project_id": inputs["project_id"]})}

session_1_thread = copilot_cfg["configurable"]["thread_id"]
session_2_thread = "demo-copilot-session-2"
assert session_1_thread != session_2_thread

recalled = read_project_memory.invoke(
    {"project_id": "brightpath-capstone"},
    {"configurable": {"thread_id": session_2_thread}},
)["memories"]

kv("written in thread", session_1_thread)
kv("read from thread", session_2_thread)
kv("decisions recalled", len(recalled))

step("recalled decisions")
for m in recalled:
    print(f"  {m['decision']:<8} {m['requirement_id']:<10} {(m['final_text'] or '—')[:44]}")

assert len(recalled) > 0, "long-term memory did not survive the thread change"
assert {m["requirement_id"] for m in recalled} == {f["requirement_id"] for f in resumed["findings"]}
ok("decisions written in one thread are readable from a different thread")


12.6  CROSS-THREAD MEMORY
  written in thread       demo-copilot-session-1
  read from thread        demo-copilot-session-2
  decisions recalled      8

--- recalled decisions -------------------------------------------------------
  approve  GAP-002    The system shall generate a weekly enrollmen
  edit     REQ-007    The system shall have an uptime of 99.9% and
  reject   REQ-007    —
  approve  GAP-001    The system shall notify a student when a sea
  approve  GAP-003    The system shall allow a student to add thei
  reject   REQ-004    —
  approve  REQ-005    Modify REQ-005 to include an exception for d
  edit     REQ-004    The system shall respond to course search qu
[PASS] decisions written in one thread are readable from a different thread


### 12.7 — Error handling

In [23]:
banner("12.7  error handling")

step("strategy 1 — retry policy attached")
_llm_tasks = {
    "supervisor_route": supervisor_route,
    "completeness_reviewer": completeness_reviewer,
    "ambiguity_reviewer": ambiguity_reviewer,
    "conflict_reviewer": conflict_reviewer,
    "testability_standards_reviewer": testability_standards_reviewer,
}
for _name, _t in _llm_tasks.items():
    assert getattr(_t, "retry_policy", None), f"{_name} has no retry policy attached"
    kv(_name, "attached", width=34)

kv("policy", f"{REVIEWER_RETRY.max_attempts} attempts, {REVIEWER_RETRY.initial_interval}s "
             f"x{REVIEWER_RETRY.backoff_factor}, cap {REVIEWER_RETRY.max_interval}s", width=34)

# The predicate is what separates a retry policy from a blind loop.
assert _is_transient_error(TimeoutError("slow")) is True
assert _is_transient_error(ValueError("bad request")) is False
kv("predicate", "retries timeouts/rate-limits, refuses permanent errors", width=34)


step("strategy 2 — fallback degrades instead of crashing")

def with_fallback(reviewer_task, label):
    """Return a structured finding if a reviewer dies, so one failure cannot kill the review."""
    def safe(*args, **kwargs):
        try:
            return reviewer_task(*args, **kwargs).result()
        except Exception as exc:
            return [ReviewFinding(
                requirement_id="REVIEWER-ERROR", issue_type="untestable", severity="low",
                reason=f"{label} failed after all retries: {type(exc).__name__}: {exc}",
                suggested_change="Manual review required — the automated reviewer was unavailable.",
            )]
    return safe


@task(retry_policy=REVIEWER_RETRY)
def _always_failing_reviewer(requirements: list[Requirement]) -> list[ReviewFinding]:
    raise RuntimeError("simulated permanent reviewer outage")


@entrypoint(checkpointer=checkpointer)
def _fallback_demo(inputs: dict) -> dict:
    return {"findings": [f.model_dump()
                         for f in with_fallback(_always_failing_reviewer, "Conflict Reviewer")([])]}


_fb = _fallback_demo.invoke({}, {"configurable": {"thread_id": "demo-fallback"}})["findings"][0]
kv("returned", _fb["requirement_id"], width=34)
kv("reason", wrap(_fb["reason"], 36), width=34)
assert _fb["requirement_id"] == "REVIEWER-ERROR" and "RuntimeError" in _fb["reason"]
ok("retry attached to all 5 LLM tasks; fallback degrades safely")


12.7  ERROR HANDLING

--- strategy 1 — retry policy attached ---------------------------------------
  supervisor_route                  attached
  completeness_reviewer             attached
  ambiguity_reviewer                attached
  conflict_reviewer                 attached
  testability_standards_reviewer    attached
  policy                            5 attempts, 3.0s x2.0, cap 45.0s
  predicate                         retries timeouts/rate-limits, refuses permanent errors

--- strategy 2 — fallback degrades instead of crashing -----------------------
  returned                          REVIEWER-ERROR
  reason                            Conflict Reviewer failed after all retries: RuntimeError: simulated permanent
                                    reviewer outage
[PASS] retry attached to all 5 LLM tasks; fallback degrades safely


### 12.8 — Observability

In [24]:
import time

banner("12.8  observability")

assert os.environ.get("LANGCHAIN_TRACING_V2") == "true", "tracing is not enabled"
assert os.environ.get("LANGCHAIN_API_KEY"), "LANGCHAIN_API_KEY missing — traces would 403"
kv("tracing", "on")
kv("project", os.environ.get("LANGCHAIN_PROJECT"))

timing_reqs = [
    Requirement(id="REQ-001", text="The system shall respond quickly to registration requests."),
    Requirement(id="REQ-002", text="The system shall never allow a drop after the deadline, "
                                   "except that drops within 24 hours are always allowed."),
]


@entrypoint(checkpointer=checkpointer)
def _timed_run(inputs: dict) -> dict:
    marks = {}
    for label, call in [
        ("supervisor_route", lambda: supervisor_route(None, timing_reqs)),
        ("ambiguity_reviewer", lambda: ambiguity_reviewer(timing_reqs)),
        ("conflict_reviewer (RAG)", lambda: conflict_reviewer(timing_reqs)),
        ("testability_reviewer (RAG)", lambda: testability_standards_reviewer(timing_reqs)),
    ]:
        t0 = time.perf_counter()
        call().result()
        marks[label] = time.perf_counter() - t0
    return {"marks": marks}


marks = _timed_run.invoke({}, {"configurable": {"thread_id": "demo-observability"}})["marks"]

step("per-stage latency (seconds)")
for stage, secs in sorted(marks.items(), key=lambda kv_: -kv_[1]):
    print(f"  {secs:6.2f}  {stage}")

slowest = max(marks, key=marks.get)
kv("slowest stage", f"{slowest} ({marks[slowest]:.2f}s of {sum(marks.values()):.2f}s)")
print("\n  Cross-check against the trace timeline at https://smith.langchain.com")
print(f"  project: {os.environ.get('LANGCHAIN_PROJECT')}")


12.8  OBSERVABILITY
  tracing                 on
  project                 capstone-requirements-reviewer
  [ambiguity] found 1 vague requirement(s)
  [conflict] found 0 contradiction(s)
  [testability] found 2 unverifiable requirement(s)

--- per-stage latency (seconds) ----------------------------------------------
    5.53  testability_reviewer (RAG)
    5.23  ambiguity_reviewer
    4.83  conflict_reviewer (RAG)
    4.65  supervisor_route
  slowest stage           testability_reviewer (RAG) (5.53s of 20.24s)

  Cross-check against the trace timeline at https://smith.langchain.com
  project: capstone-requirements-reviewer


### What the trace showed

Tracing was enabled with `LANGCHAIN_TRACING_V2` and the full system is visible in LangSmith
under the project `capstone-requirements-reviewer` — 86 traces, 137,381 tokens, and a 6%
error rate over the last 7 days. Three things the trace told us that the notebook output
alone did not:

**1. Retrieval is free; the LLM calls are the entire cost.** The `search_requirements_standards`
spans complete in 0.02–0.03s and `parse_requirements` in 0.69–0.73s, while a single
`LangGraph` pipeline run takes 15–25s. Our first instinct had been that the RAG-grounded
reviewers were slow because of retrieval — the trace shows retrieval is negligible and the
latency is entirely the Groq calls. The right lever for speed is fewer or smaller LLM calls,
not a faster vector store.

**2. The retry policy is visibly firing.** Four traces carry `RateLimitError` with latencies
of 47.94s, 48.74s, 105.94s and 119.56s. Those durations are the retry backoff itself: the
policy re-attempted with exponential delay before the error finally surfaced. A single failed
API call takes under a second, so a two-minute span is the retry mechanism working, not a
slow request.

**3. P50 vs P99 quantifies the split.** P50 latency is 1.10s but P99 is 105.12s. The median
span is a cheap one (retrieval, a memory write, a synthesizer step) while the tail is
dominated by the rate-limited runs above. Those two numbers being two orders of magnitude
apart is the clearest signal that the bottleneck is external quota, not our graph.

**What we changed as a result.** The rate limiter in §3 was added directly in response to
these traces, and the `_is_transient_error` predicate now returns `False` for daily-quota
(`TPD`) errors — the trace showed us spending nearly two minutes retrying a limit that
cannot clear inside a retry window.